# 04 — Inferenza dei Layer Normativi e Heatmap di Ibridità

Questo notebook implementa il cuore metodologico del progetto: **i livelli gerarchici
non sono predefiniti ma emergono dai dati** della specifica materia analizzata.

## Pipeline in quattro fasi

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Segmenti di ogni atto | LLM 1: descrizione funzionale contestualizzata per segmento | `segments_descriptions.csv` |
| **B** | Descrizioni funzionali | Embedding + UMAP + HDBSCAN → cluster = layer emergenti | — |
| **B.2** | 10 descrizioni representative per cluster | LLM 2: nome del layer | `layer_mapping.csv` |
| **C** | Articoli + layer noti | LLM 3: distribuzione % articolo × layer | `nodes_heatmap.csv` |
| **D** | Matrice per atto | Entropia media → score di ibridità continuo | `nodes_hybridity.csv` |

## Principio metodologico

Il clustering avviene a livello di **segmento** (articolo o considerando), non di atto.
I cluster che emergono rappresentano i livelli gerarchici specifici per quella materia —
quanti siano lo decide l'algoritmo, non l'analista.

Un atto è **ibrido** se i suoi articoli hanno distribuzioni molto diverse tra loro:
alcuni concentrati su livelli apicali, altri su livelli tecnici di dettaglio.
L'ibridità è misurata come **entropia media degli articoli**.

## Output

| File | Contenuto |
|---|---|
| `segments_descriptions.csv` | Una riga per segmento con la descrizione funzionale (LLM 1) |
| `layer_mapping.csv` | Cluster → nome layer con descrizione e rank gerarchico |
| `nodes_heatmap.csv` | Una riga per (celex, articolo) con % per ogni layer |
| `nodes_hybridity.csv` | Una riga per atto con score ibridità e layer dominante |

---

> **Nota sul costo API**: Fase A chiama l'LLM una volta per segmento, Fase C una volta
> per articolo. Con ~200 atti e ~20 segmenti/atto = ordine di 4.000–6.000 chiamate.
> Il checkpointing granulare permette di riprendere da dove si era interrotti.

## 0. Configurazione

**Modifica solo questa cella.** Il resto del notebook gira in automatico.

In [3]:
import re, json
import pandas as pd
from pathlib import Path

out = Path('../data/output/appalti_it')

# ── Regex per segmentare il testo italiano in articoli ────────────────────────
RE_ART_IT = re.compile(
    r'(?m)^[ \t]*Art(?:icolo)?\.?\s+'
    r'(\d+(?:\s*-?\s*(?:bis|ter|quater|quinquies|sexies|septies|octies|novies|decies))?)'
    r'[ \t]*\.?[ \t]*(?:\(([^)\n]{0,120})\))?[ \t]*$',
    re.IGNORECASE
)

def build_segments_it(full_text):
    """Segmenta full_text italiano in articoli."""
    if not full_text or str(full_text) == 'nan':
        return []
    text = str(full_text)
    segs = []
    splits = list(RE_ART_IT.finditer(text))
    if splits:
        pre = text[:splits[0].start()].strip()
        if len(pre) >= 30:
            segs.append({'tipo': 'preambolo_header', 'identificatore': 'preambolo', 'testo': pre[:3000]})
    for i, m in enumerate(splits):
        art_num = m.group(1).strip()
        rubrica = (m.group(2) or '').strip()
        start   = m.start()
        end     = splits[i+1].start() if i+1 < len(splits) else len(text)
        testo   = text[start:end].strip()
        if len(testo) >= 30:
            segs.append({'tipo': 'articolo', 'identificatore': art_num,
                         'testo': testo[:10000], 'rubrica': rubrica})
    return segs


# ── Carica e prepara atti IT ───────────────────────────────────────────────────
df_it_raw = pd.read_csv(out / 'nodes_texts_it.csv')
df_it_raw = df_it_raw.loc[:, ~df_it_raw.columns.duplicated()]

rows_it = []
for _, row in df_it_raw.iterrows():
    full_text = str(row.get('full_text', '')) if pd.notna(row.get('full_text', '')) else ''
    segs      = build_segments_it(full_text)
    n_art     = sum(1 for s in segs if s['tipo'] == 'articolo')
    status    = 'no_text' if not full_text or full_text == 'nan' \
                else ('no_segments' if n_art == 0 else 'ok')
    rows_it.append({
        'Id':           row.get('id',    row.get('slug', '')),
        'Label':        row.get('label', row.get('id',   '')),
        'title':        row.get('titolo', row.get('title', '')),
        'segments':     json.dumps(segs, ensure_ascii=False),
        'text_status':  status,
        'layer_atteso': row.get('layer_atteso', ''),
        'giurisdizione': 'IT',
        'n_articoli':   n_art,
        'quality_score': row.get('quality_score', 0),
    })

df_it = pd.DataFrame(rows_it)
print(f"IT: {len(df_it)} atti  |  ok={df_it[df_it.text_status=='ok'].shape[0]}")


# ── Carica atti EU (segments già pronti dal fetch EUR-Lex) ────────────────────
eu_path = out / 'nodes_texts_eu_appalti.csv'
if eu_path.exists():
    df_eu_raw = pd.read_csv(eu_path)
    rows_eu = []
    for _, row in df_eu_raw.iterrows():
        segs_raw = row.get('segments', '')
        segs     = json.loads(str(segs_raw)) if segs_raw and str(segs_raw) not in ('nan', '[]', '') else []
        n_art    = sum(1 for s in segs if s['tipo'] == 'articolo')
        full_text = str(row.get('full_text', '') or '')
        status   = 'no_text' if not full_text or full_text == 'nan' \
                   else ('no_segments' if n_art == 0 else 'ok')
        rows_eu.append({
            'Id':           row.get('celex', ''),
            'Label':        row.get('celex', ''),
            'title':        row.get('title', ''),
            'segments':     json.dumps(segs, ensure_ascii=False),
            'text_status':  status,
            'layer_atteso': row.get('layer_atteso', ''),
            'giurisdizione': 'EU',
            'n_articoli':   n_art,
            'quality_score': row.get('quality_score', 0),
        })
    df_eu = pd.DataFrame(rows_eu)
    print(f"EU: {len(df_eu)} atti  |  ok={df_eu[df_eu.text_status=='ok'].shape[0]}")
    df = pd.concat([df_it, df_eu], ignore_index=True)
else:
    df = df_it
    print("nodes_texts_eu_appalti.csv non trovato — solo IT")

print(f"Totale: {len(df)}  |  ok={df[df.text_status=='ok'].shape[0]}")
df.to_csv(out / 'nodes_texts.csv', index=False)
print(f"✓ Salvato nodes_texts.csv — colonne: {list(df.columns)}")

IT: 22 atti  |  ok=17
EU: 6 atti  |  ok=4
Totale: 28  |  ok=21
✓ Salvato nodes_texts.csv — colonne: ['Id', 'Label', 'title', 'segments', 'text_status', 'layer_atteso', 'giurisdizione', 'n_articoli', 'quality_score']


In [5]:
MATERIA_NAME = "appalti_it"

TEMA_DESCRIZIONE = (
    "Appalti pubblici e contratti pubblici in Italia. "
    "Disciplina delle procedure di gara, dei requisiti delle stazioni appaltanti, "
    "delle concessioni e degli affidamenti di lavori, servizi e forniture pubblici."
)


# ── Modello OpenAI ─────────────────────────────────────────────────────────────
LLM_MODEL = "gpt-5.4-mini"   


# ── Parametri chiamate API ────────────────────────────────────────────────────
LLM_MAX_TOKENS_A   = 300    # Fase A: descrizione livello di astrazione (2-3 frasi)
LLM_MAX_TOKENS_A2  = 800    # Fase A2: JSON distribuzione % Lamfalussy
LLM_MAX_TOKENS_B2  = 400    # Fase B.2: JSON nome + descrizione layer (legacy, non usato)
LLM_MAX_TOKENS_C   = 1500    # Fase C: JSON percentuali
LLM_DELAY_SECONDS  = 0.3    # pausa tra chiamate (rispetta il rate limit)
LLM_MAX_RETRIES    = 3      # tentativi in caso di errore transitorio
LLM_RETRY_DELAY    = 5.0    # secondi tra retry


# ── Parametri checkpoint ──────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 100    # segmenti/articoli tra un salvataggio e il successivo


# ── Parametri induzione layer (Fase B) ────────────────────────────────────────
N_SAMPLE_DESCRIPTIONS = 200   # descrizioni campionate per l'induzione dei layer


## 1. Import e Percorsi

In [6]:
import os
import json
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path  = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

INPUT_FILE              = os.path.join(output_path, 'nodes_texts.csv')
EDGES_FILE              = os.path.join(output_path, 'edges_focal.csv')
SEGMENTS_DESC_FILE      = os.path.join(output_path, 'segments_descriptions.csv')
LAYER_MAPPING_FILE      = os.path.join(output_path, 'layer_mapping.csv')
NODES_HEATMAP_FILE      = os.path.join(output_path, 'nodes_heatmap.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')
HEATMAP_CKPT_FILE       = os.path.join(output_path, 'heatmap_checkpoint.csv')

# ── path ESISTENTI (4 livelli) ───────────────────────────────────────────────
SEGMENTS_LAMF_FILE      = os.path.join(output_path, 'segments_lamfalussy.csv')
SEGMENTS_LAMF_CKPT_FILE = os.path.join(output_path, 'segments_lamfalussy_checkpoint.csv')
NODES_LAMFALUSSY_FILE   = os.path.join(output_path, 'nodes_lamfalussy.csv')

# ── path NUOVI (2 livelli) ─────────────────────────────────────────────────
SEGMENTS_LAMF2_FILE      = os.path.join(output_path, 'segments_lamfalussy2.csv')
SEGMENTS_LAMF2_CKPT_FILE = os.path.join(output_path, 'segments_lamfalussy2_checkpoint.csv')
NODES_LAMFALUSSY2_FILE   = os.path.join(output_path, 'nodes_lamfalussy2.csv')

print(f"Materia:       {MATERIA_NAME}")
print(f"Input:         {INPUT_FILE}")
print(f"Modello LLM:   {LLM_MODEL}")

Materia:       appalti_it
Input:         ..\data\output\appalti_it\nodes_texts.csv
Modello LLM:   gpt-5.4-mini


## 2. Caricamento Dati

In [7]:
import re, json
import pandas as pd

nodes = pd.read_csv(INPUT_FILE)

# ── Check finale ──────────────────────────────────────────────────────────────
print(f"Nodi totali: {len(nodes)}")
nodes_ok   = nodes[nodes['text_status'] == 'ok'].copy()
nodes_fail = nodes[nodes['text_status'] != 'ok'].copy()
print(f"Atti con testo (text_status=ok):  {len(nodes_ok)}")
print(f"Atti senza testo (esclusi):       {len(nodes_fail)}")
if len(nodes_ok):
    print(f"\nDistribuzione per giurisdizione (solo ok):")
    print(nodes_ok['giurisdizione'].value_counts().to_string())
print(f"\nDistribuzione text_status:")
print(nodes['text_status'].value_counts().to_string())

Nodi totali: 28
Atti con testo (text_status=ok):  21
Atti senza testo (esclusi):       7

Distribuzione per giurisdizione (solo ok):
giurisdizione
IT    17
EU     4

Distribuzione text_status:
text_status
ok             21
no_segments     5
no_text         2


## 3. Divisione in Segmenti

La colonna `segments` di ogni atto contiene una lista JSON di segmenti strutturati
(articoli, considerando, allegati). Questa cella costruisce un DataFrame flat
`segments_df` con **una riga per segmento** — l'unità di analisi del clustering.

In [8]:
def parse_segments(row):
    """Parsa la colonna 'segments' e restituisce lista di dict arricchiti."""
    celex = row.get('Label', row['Id'])
    title = str(row.get('title', ''))
    raw   = row.get('segments')

    if pd.isna(raw) or not str(raw).strip():
        return []
    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        return []

    result = []
    for s in segs:
        result.append({
            'celex':          celex,
            'node_id':        row['Id'],
            'title_atto':     title,
            'tipo':           s.get('tipo', ''),
            'identificatore': s.get('identificatore', ''),
            'testo':          s.get('testo', ''),
        })
    return result


all_segments = []
for _, row in nodes_ok.iterrows():
    all_segments.extend(parse_segments(row))

segments_df = pd.DataFrame(all_segments)

# ID univoco per segmento
segments_df['segment_id'] = (
    segments_df['celex'] + '__' +
    segments_df['tipo'] + '__' +
    segments_df['identificatore'].astype(str)
)

# Rimuove duplicati su segment_id (stesso atto, stesso tipo, stesso identificatore)
before_dedup = len(segments_df)
segments_df = segments_df.drop_duplicates(subset='segment_id', keep='first').reset_index(drop=True)

# Rimuove segmenti con testo troppo breve per essere informativi
MIN_TESTO_LEN = 30
before = len(segments_df)
segments_df = segments_df[segments_df['testo'].str.len() >= MIN_TESTO_LEN].reset_index(drop=True)

# Esclude considerando e header preambolo — non entrano nel clustering né nella heatmap
TIPI_ESCLUSI = {'considerando', 'preambolo_header'}
before_tipi = len(segments_df)
segments_df = segments_df[~segments_df['tipo'].isin(TIPI_ESCLUSI)].reset_index(drop=True)

print(f"Segmenti estratti:              {before_dedup:,}")
print(f"Segmenti scartati (duplicati):  {before_dedup - before:,}")
print(f"Segmenti validi (>={MIN_TESTO_LEN} car): {len(segments_df):,}")
print(f"Segmenti scartati (testo):      {before - before_tipi:,}")
print(f"Segmenti scartati (tipo):       {before_tipi - len(segments_df):,}")
print()
print("Distribuzione per tipo:")
print(segments_df['tipo'].value_counts().to_string())
print()
print(f"Segmenti medi per atto:  {segments_df.groupby('celex').size().mean():.1f}")

Segmenti estratti:              2,672
Segmenti scartati (duplicati):  739
Segmenti validi (>=30 car): 1,426
Segmenti scartati (testo):      0
Segmenti scartati (tipo):       507

Distribuzione per tipo:
tipo
articolo    1426

Segmenti medi per atto:  67.9


## 4. Fase A — Livello di Astrazione per Segmento (LLM 1)

Per ogni segmento l'LLM produce una **descrizione libera del livello di astrazione**:
dove si colloca il segmento nella piramide normativa — quanto è fondazionale vs tecnico/operativo.

La descrizione è volutamente domain-agnostic e non usa la terminologia Lamfalussy:
non vengono imposti tag o categorie. I livelli specifici per materia emergeranno
liberamente dal clustering in Fase B.

**Fase A2** (successiva) assegnerà poi i livelli Lamfalussy standard (L1–L4)
usando queste descrizioni come contesto.

> **Checkpoint**: i risultati vengono salvati in `segments_descriptions.csv` ogni
> `CHECKPOINT_EVERY` segmenti. Rieseguire la cella riprende dal punto di interruzione.

In [26]:
def build_prompt_functional_description(testo, tipo, identificatore, title_atto, tema):
    return f"""You are an expert in Italian and European law, 
    with knowledge of EU legislation and its transposition into Italian law.

Your task is to describe the position of this legal segment in the regulatory
hierarchy — how general or specific it is, and why.

Focus exclusively on the ABSTRACTION LEVEL:
Does this segment state a broad principle that governs the entire framework,
or does it implement a narrow technical detail that only applies in a specific
situation? Where on the spectrum from foundational to operational does it sit?

Describe:
1. Where it sits on the spectrum (foundational / structural / operational / technical)
2. Why — what feature of the text places it there (e.g., it establishes a purpose,
   it delegates a power, it specifies a procedure, it fixes a threshold,
   it sets an effective date, it defines a concept)

## Critical rules
- Do NOT describe what the segment is about thematically.
- Do NOT mention the specific subject matter (FDI, data protection, banking,
  subsidies, etc.).
- Describe only the position in the normative hierarchy and its structural reason.
- 2-3 sentences maximum.

## Examples of correct descriptions
- "Foundational recital that states the overarching policy rationale justifying
   the entire legislative intervention — sits at the highest level of abstraction
   as it frames the purpose of the whole framework without specifying any operative rule."
- "Structural empowerment clause delegating secondary rule-making power to an
   institution within defined substantive limits — sits at an intermediate level,
   organising institutional competences without specifying how they must be exercised."
- "Narrow operative criterion specifying one concrete factor to be checked during
   a particular assessment — highly specific, functioning as a technical sub-rule
   within a broader procedure."
- "Terminal commencement clause fixing the date the act becomes legally effective —
   sits at the most technical end, containing no substantive normative content."

## Also avoid
- Descriptions so vague they say nothing: "Sets out a provision within the framework."
- Restating what the text says without identifying the hierarchical position.
- Any reference to the subject matter of the regulation.

Act title: {title_atto}
Segment ({tipo} {identificatore}):
{testo}

Reply ONLY with the description (2-3 sentences), in English, no other text."""

def call_llm(client, prompt, max_tokens):
    """
    Chiama l'LLM OpenAI con retry automatico su errori transitori.
    Restituisce (testo_risposta, status) dove status è 'ok' | 'error' | 'empty'.
    """
    for attempt in range(LLM_MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                max_completion_tokens=max_tokens,
                temperature=0.3,
                messages=[{'role': 'user', 'content': prompt}]
            )
            text = response.choices[0].message.content.strip()
            if not text:
                return '', 'empty'
            return text, 'ok'

        except openai.RateLimitError:
            wait = LLM_RETRY_DELAY * (attempt + 1) * 2
            print(f"  [RateLimit] attesa {wait:.0f}s (tentativo {attempt+1}/{LLM_MAX_RETRIES})")
            time.sleep(wait)

        except openai.APIStatusError as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

        except Exception as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

    return 'ERROR: max retries exceeded', 'error'


print("Funzioni LLM definite.")

Funzioni LLM definite.


In [31]:
# ── TEST su 5 segmenti ────────────────────────────────────────────────────────
'''from openai import OpenAI

client = OpenAI()


test_segments = segments_df.sample(5, random_state=42)

for _, seg in test_segments.iterrows():
    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    
    print(f"CELEX: {seg['celex']}  |  {seg['tipo']} {seg['identificatore']}")
    print(f"Testo: {seg['testo'][:150]}...")
    print(f"→ {descrizione}")
    print()'''

CELEX: dlgs_36_2023  |  articolo 124
Testo: Articolo 124.
Esecuzione o completamento dei lavori, servizi o forniture nel caso di procedura di insolvenza o di 
impedimento alla prosecuzione dell’...
→ This is an operational-to-technical provision: it lays down a detailed contingency mechanism for a specific set of exceptional procedural situations, rather than a general organizing principle. Its abstraction level is low because it specifies who must be consulted, in what order, under what conditions, and with which exceptions and time limits, leaving little room for further normative interpretation.

CELEX: 32014L0023  |  articolo 23
Testo: Articolo 23 Concessioni riguardanti sia attività cui all’allegato II sia attività con aspetti di difesa o di sicurezza 1. Nel caso di contratti destin...
→ Operational to technical provision: it does not set a broad governing principle, but instead lays down a conditional decision rule for how a contracting authority must structure and classify a mix

In [9]:
# ── Gestione checkpoint ───────────────────────────────────────────────────────
if os.path.exists(SEGMENTS_DESC_FILE):
    segs_done = pd.read_csv(SEGMENTS_DESC_FILE)
    done_ids  = set(segs_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_ids):,} segmenti già descritti.")
else:
    segs_done = pd.DataFrame()
    done_ids  = set()
    print("Nessun checkpoint — si parte da zero.")

segments_todo = segments_df[~segments_df['segment_id'].isin(done_ids)].copy()
print(f"Da descrivere: {len(segments_todo):,}")

if len(segments_todo) == 0:
    print("✓ Tutti i segmenti già descritti — si può passare alla Fase B.")

Checkpoint trovato: 1,426 segmenti già descritti.
Da descrivere: 0
✓ Tutti i segmenti già descritti — si può passare alla Fase B.


In [10]:
%%time
from openai import OpenAI

client = OpenAI()

new_rows = []
n_ok = n_error = 0
total = len(segments_todo)

for i, (_, seg) in enumerate(segments_todo.iterrows()):

    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    if status != 'ok':
        print(f"  [{i+1}] ERRORE: {descrizione}")

    new_rows.append({
        'segment_id':              seg['segment_id'],
        'celex':                   seg['celex'],
        'node_id':                 seg['node_id'],
        'tipo':                    seg['tipo'],
        'identificatore':          seg['identificatore'],
        'testo_originale':         seg['testo'],
        'descrizione_funzionale':  descrizione,
        'llm_status':              status,
    })

    if status == 'ok':
        n_ok += 1
    else:
        n_error += 1

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([segs_done, batch], ignore_index=True) if not segs_done.empty else batch
        combined.to_csv(SEGMENTS_DESC_FILE, index=False)
        print(f"  [{i+1:>5}/{total}]  {(i+1)/total*100:5.1f}%   ok: {n_ok}   errori: {n_error}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE A — ok: {n_ok:,}   errori: {n_error:,}")
print("=" * 50)


FASE A — ok: 0   errori: 0
CPU times: total: 297 ms
Wall time: 347 ms


## 4b. Fase A2 — Assegnazione Livelli Lamfalussy per Segmento (LLM 2)

Partendo dall'output di Fase A (`segments_descriptions.csv`), per ogni segmento
l'LLM assegna una distribuzione percentuale sui 4 livelli Lamfalussy.

La **descrizione del livello di astrazione** prodotta in Fase A viene fornita come
contesto: l'LLM non deve più dedurre la posizione gerarchica dal testo,
deve solo mappare quella descrizione sulla tassonomia Lamfalussy standard.

**Output**: `segments_lamfalussy.csv` — una riga per segmento con
`lamf_L1`, `lamf_L2`, `lamf_L3`, `lamf_L4` (somma = 100).

> Gira su **tutti** i segmenti (considerando + articoli + allegati).
> Fase C2 filtrerà ai soli articoli per il calcolo dell'entropia.

In [13]:
# ── Costanti Lamfalussy — versione a 2 livelli ─────────────────────────────
import json

LAMFALUSSY_LEVELS = [
    {
        'key':         'L1',
        'name':        'Level 1 — Framework principles',
        'description': (
            'Level 1 legislation sets out the core framework principles and defines essential features: '
            'objectives of the regulation; scope and key definitions; main rights and obligations; '
            'institutional framework; core harmonization choices; delegation clauses empowering '
            'the Commission or regulatory agencies to adopt implementing measures. '
            'These elements reflect fundamental political choices and are intended to be stable and enduring.'
        ),
    },
    {
        'key':         'L2',
        'name':        'Level 2 — Operational rules',
        'description': (
            'Level 2 legislation translates Level 1 principles into operational rules by specifying '
            'their technical content and modes of application: rules for procedures; specifications '
            'of quantitative benchmarks; mechanisms for verifying and enforcing compliance; '
            'timelines and reporting frequencies; standards for consistent application across jurisdictions. '
            'Unlike Level 1, Level 2 is designed to be flexible, allowing continuous technical '
            'adjustments without reopening primary legislation.'
        ),
    },
]

LAMF_KEYS = [l['key'] for l in LAMFALUSSY_LEVELS]   # ['L1', 'L2']
LAMF_COLS = [f'lamf_{k}' for k in LAMF_KEYS]        # ['lamf_L1', 'lamf_L2']


def build_prompt_lamfalussy_assignment(testo, identificatore, tipo):
    return f"""You are classifying provisions of EU legislation.

Classification framework:

Level 1 (L1):
Level 1 legislation sets out the core framework principles and defines essential features. Level 1 includes, for example, the objectives of the regulation; its scope and key definitions; the main rights and obligations; the institutional framework; core choices regarding harmonization between national and European levels; and the delegation clauses empowering the Commission or regulatory agencies to adopt implementing measures. These elements reflect fundamental political choices and are intended to be stable and enduring.

Level 2 (L2):
Level 2 legislation translates Level 1 principles into operational rules by specifying their technical content and modes of application. Level 2 includes, for example, rules for procedures; specifications of quantitative benchmarks; mechanisms for verifying and enforcing compliance; timelines and reporting frequencies; and standards for ensuring consistent application across jurisdictions. Unlike Level 1, Level 2 is designed to be flexible, allowing for continuous technical adjustments without reopening primary legislation.

Decision rules:
Classify as L1 if the provision primarily:
- defines concepts, scope, or objectives
- establishes general rights or obligations
- sets institutional roles or powers
- contains delegation clauses

Classify as L2 if the provision primarily:
- specifies procedures or processes
- defines technical or operational requirements
- sets quantitative thresholds or benchmarks
- describes reporting, monitoring, or enforcement

Conflict resolution:
- If both L1 and L2 elements are present, assign percentages reflecting their relative weight (e.g. 70 L1 / 30 L2).
- If no dominant function can be identified, assign 50 to each.
- Percentages must always sum to exactly 100.

Segment ({tipo} {identificatore}):
{testo}

Reply ONLY with valid JSON IN ENGLISH (no backticks):
{{
  "L1": X, "L2": X,
  "evidence": [
    {{"testo": "exact short quote from segment", "layer": "L1", "motivo": "one sentence explanation"}},
    ...
  ]
}}"""


print("Costanti Lamfalussy e funzioni Fase A2 definite.")
print(f"Livelli: {LAMF_KEYS}")

Costanti Lamfalussy e funzioni Fase A2 definite.
Livelli: ['L1', 'L2']


In [11]:
%%time

def parse_percentage_response(response_text, keys):
    import json, re
    text = response_text.strip()
    text = re.sub(r'^```[a-z]*\n?', '', text)
    text = re.sub(r'\n?```$', '', text)
    try:
        data = json.loads(text)
        result = {}
        for k in keys:
            val = float(data.get(k, 0))
            result[k] = max(0.0, min(100.0, val))
        total = sum(result.values())
        if total > 0 and abs(total - 100) > 1:
            result = {k: v / total * 100 for k, v in result.items()}
        result['_evidence'] = data.get('evidence', [])
        return result
    except Exception:
        return None

# ── Gestione checkpoint ────────────────────────────────────────────────────────
# Input: articles_df (900 articoli interi, non paragrafi)
done_lamf_ids = set()
if os.path.exists(SEGMENTS_LAMF2_CKPT_FILE):
    lamf_segs_done = pd.read_csv(SEGMENTS_LAMF2_CKPT_FILE)
    done_lamf_ids  = set(lamf_segs_done['segment_id'])
    print(f"Checkpoint: {len(done_lamf_ids):,} già classificati.")
else:
    lamf_segs_done = pd.DataFrame()

articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()

segs_a_todo = articles_df[~articles_df['segment_id'].isin(done_lamf_ids)].copy()
print(f"Articoli da classificare: {len(segs_a_todo):,}")

new_lamf_rows = []
n_ok_a2 = n_error_a2 = 0
total_a2 = len(segs_a_todo)

for i, (_, seg) in enumerate(segs_a_todo.iterrows()):

    prompt = build_prompt_lamfalussy_assignment(
        testo          = seg['testo'],
        identificatore = seg['identificatore'],
        tipo           = seg['tipo'],
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_A2)

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, LAMF_KEYS)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':    seg['segment_id'],
        'celex':         seg['celex'],
        'node_id':       seg['node_id'],
        'tipo':          seg['tipo'],
        'identificatore': seg['identificatore'],
        'llm_status':    status,
        'evidence':      json.dumps(distribution.get('_evidence', []),
                                    ensure_ascii=False) if distribution else '[]',
    }
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        row[col] = round(distribution[key], 2) if distribution else 0.0

    if distribution:
        n_ok_a2 += 1
    else:
        n_error_a2 += 1

    new_lamf_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_a2:
        batch    = pd.DataFrame(new_lamf_rows)
        combined = pd.concat([lamf_segs_done, batch], ignore_index=True) if not lamf_segs_done.empty else batch
        combined.to_csv(SEGMENTS_LAMF2_CKPT_FILE, index=False)
        print(f"  [{i+1:>5}/{total_a2}]  {(i+1)/total_a2*100:5.1f}%   ok: {n_ok_a2}   errori: {n_error_a2}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print('=' * 50)
print(f"FASE A2 — ok: {n_ok_a2:,}   errori: {n_error_a2:,}")
print('=' * 50)


Checkpoint: 1,434 già classificati.
Articoli da classificare: 0

FASE A2 — ok: 0   errori: 0
CPU times: total: 31.2 ms
Wall time: 24.6 ms


In [14]:
# Salva output Fase A2
segments_lamf_final = pd.read_csv(SEGMENTS_LAMF2_CKPT_FILE)
segments_lamf_final.to_csv(SEGMENTS_LAMF2_FILE, index=False)

print(f"Salvato: {SEGMENTS_LAMF2_FILE}")
print(f"Segmenti totali: {len(segments_lamf_final):,}  |  Colonne Lamfalussy: {LAMF_COLS}")
print()

ok_mask = segments_lamf_final['llm_status'] == 'ok'
print("Distribuzione media % per livello Lamfalussy (tutti i segmenti ok):")
for col, key in zip(LAMF_COLS, LAMF_KEYS):
    level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
    mean_pct = segments_lamf_final.loc[ok_mask, col].mean()
    bar      = '█' * int(mean_pct / 2)
    print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")

print()
# Breakdown per tipo di segmento
print("Distribuzione per tipo di segmento:")
for tipo in ['considerando', 'articolo', 'allegato']:
    n = ok_mask & (segments_lamf_final['tipo'] == tipo)
    if n.sum() > 0:
        print(f"  {tipo:>12}: {n.sum():>4} segmenti")


Salvato: ..\data\output\appalti_it\segments_lamfalussy2.csv
Segmenti totali: 1,434  |  Colonne Lamfalussy: ['lamf_L1', 'lamf_L2']

Distribuzione media % per livello Lamfalussy (tutti i segmenti ok):
  Level 1 — Framework principles................  42.1%  █████████████████████
  Level 2 — Operational rules...................  57.9%  ████████████████████████████

Distribuzione per tipo di segmento:
      articolo: 1433 segmenti


## 5. Fase B — Induzione dei Layer Normativi via LLM

I livelli gerarchici specifici per la materia vengono **indotti dall'LLM** a partire
da un campione stratificato delle descrizioni funzionali prodotte in Fase A.

A differenza dell'approccio precedente basato su `embedding + UMAP + HDBSCAN`, qui
una **singola chiamata LLM** legge un campione rappresentativo del corpus e produce
direttamente la tassonomia gerarchica — ordinata dal più fondazionale al più tecnico,
con nome, descrizione e `layer_id` (slug) per ogni livello.

L'LLM decide autonomamente quanti livelli esistono nel corpus: il prompt non impone
un numero target.

> **Output**: `layer_mapping.csv` con colonne `layer_rank`, `layer_name`,
> `layer_description`, `layer_id`.

> **Compatibilità con file legacy**: la cella di preparazione (5.1) rileva
> automaticamente i CSV prodotti dalla pipeline di clustering precedente
> (`layer_mapping.csv` privo di `layer_id`, `heatmap_checkpoint.csv` e
> `nodes_heatmap.csv` con colonne `pct__` basate sui nomi-clustering) e li rimuove
> per forzare la rigenerazione coerente con la nuova struttura.


In [ ]:
# ── 5.1 Detection dei file legacy + campionamento stratificato ────────────────
import shutil

def detect_and_clean_legacy(layer_mapping_path, heatmap_ckpt_path, heatmap_path):
    if not os.path.exists(layer_mapping_path):
        return False
    try:
        cols = pd.read_csv(layer_mapping_path, nrows=0).columns.tolist()
    except Exception as e:
        print(f"  WARN: impossibile leggere {layer_mapping_path} ({e}) — non rimuovo nulla")
        return False
    if 'layer_id' in cols:
        return False
    print("Rilevato layer_mapping.csv legacy (senza colonna 'layer_id').")
    print("Rimuovo i file di pipeline incompatibili per forzare la rigenerazione:")
    for p in [layer_mapping_path, heatmap_ckpt_path, heatmap_path]:
        if os.path.exists(p):
            os.remove(p)
            print(f"  ✓ rimosso {p}")
    return True

_legacy_cleaned = detect_and_clean_legacy(
    LAYER_MAPPING_FILE, HEATMAP_CKPT_FILE, NODES_HEATMAP_FILE
)
if _legacy_cleaned:
    print()


# ── Carica output Fase A ──────────────────────────────────────────────────────
segs_a_out = pd.read_csv(SEGMENTS_DESC_FILE)
segs_a_ok  = segs_a_out[segs_a_out['llm_status'] == 'ok'].copy()
print(f"Segmenti disponibili per campionamento: {len(segs_a_ok):,}")


# ── Campionamento stratificato per celex ──────────────────────────────────────
# Prende un numero proporzionale di segmenti per ogni atto, così il campione
# copre il corpus e non è dominato dagli atti più lunghi.

celex_col = 'celex' if 'celex' in segs_a_ok.columns else 'node_id'

def stratified_sample(df, group_col, n_total, random_state=42):
    counts = df.groupby(group_col).size()
    fracs  = (counts / counts.sum() * n_total).round().astype(int).clip(lower=1)
    sampled = []
    for grp, cnt in fracs.items():
        subset = df[df[group_col] == grp]
        sampled.append(subset.sample(min(cnt, len(subset)), random_state=random_state))
    return pd.concat(sampled).sample(frac=1, random_state=random_state)

sample_df    = stratified_sample(segs_a_ok, celex_col, N_SAMPLE_DESCRIPTIONS)
descriptions = sample_df['descrizione_funzionale'].tolist()
print(f"Campione: {len(descriptions)} descrizioni da {sample_df[celex_col].nunique()} atti")


Segmenti disponibili per campionamento: 1,426
Campione: 199 descrizioni da 21 atti


In [16]:
# ── 5.2 Prompt di induzione + chiamata LLM con checkpoint ────────────────────

def build_prompt_layer_induction(descriptions_sample, tema):
    desc_list = '\n'.join([f"  {i+1}. {d}" for i, d in enumerate(descriptions_sample)])
    return f"""You are an expert in Italian and European law,
with knowledge of EU legislation and its transposition into Italian law.

Below are {len(descriptions_sample)} functional descriptions of legal segments
drawn from a corpus of legislative acts on the following subject matter:
{tema}

Each description characterises the hierarchical position of one segment in the
regulatory pyramid — how foundational or technical it is, and why.

Your task is to identify the distinct hierarchical levels that actually structure
this body of law, by reading across all the descriptions and finding the natural
groupings in the normative hierarchy.

## What a good layer taxonomy looks like

- Layers must be defined by HIERARCHICAL FUNCTION, not by thematic content.
  Bad: "Public procurement notification rules". Good: "Operative procedural provisions".
- Layers must be ordered from most foundational (constitutional basis, enabling
  principles, scope definitions) to most technical and operational (deadlines,
  cross-references, commencement clauses, amending provisions).
- Each layer must be genuinely distinct from the others in its position in the
  normative hierarchy. Do not create layers that describe the same hierarchical
  level with different words.
- Use as many layers as the corpus genuinely requires. Do not force the
  taxonomy into a fixed number. If the corpus has 4 distinct levels, return 4.
  If it has 7, return 7. Completeness and precision matter more than brevity.
- Each layer must cover a non-trivial portion of the corpus. Do not create
  micro-layers for isolated edge cases.
- The corpus includes both Italian national law and EU directives/regulations.
  The taxonomy must capture levels that apply across both legal traditions.

## Output format

Reply ONLY with a JSON array, ordered from most foundational (index 0) to most
technical (last index). No backticks, no other text.

[
  {{
    "layer_rank": 1,
    "layer_name": "Short functional name (3-6 words)",
    "layer_description": "2-3 sentences describing the normative role and what
      kinds of provisions belong here. Be precise about the hierarchical
      position — what distinguishes this layer from adjacent ones."
  }},
  ...
]

Functional descriptions:
{desc_list}"""


# ── Checkpoint + chiamata ──────────────────────────────────────────────────────
if os.path.exists(LAYER_MAPPING_FILE):
    layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
    print(f"Checkpoint trovato: {len(layer_mapping_df)} layer già indotti.")
else:
    from openai import OpenAI
    client = OpenAI()

    prompt_induction       = build_prompt_layer_induction(descriptions, TEMA_DESCRIZIONE)
    response_text, status  = call_llm(client, prompt_induction, 2000)

    if status != 'ok':
        raise RuntimeError(f"Induzione layer fallita: {response_text}")

    clean         = response_text.replace('```json', '').replace('```', '').strip()
    layer_records = json.loads(clean)

    # Aggiungi layer_id (slug del nome, usato come chiave nelle heatmap)
    import re as _re
    seen_ids = {}
    for r in layer_records:
        slug = _re.sub(r'[^a-z0-9]+', '_', r['layer_name'].lower()).strip('_')
        # Garantisce unicità in caso di slug collidenti
        if slug in seen_ids:
            seen_ids[slug] += 1
            slug = f"{slug}_{seen_ids[slug]}"
        else:
            seen_ids[slug] = 1
        r['layer_id'] = slug

    layer_mapping_df = pd.DataFrame(layer_records)
    layer_mapping_df.to_csv(LAYER_MAPPING_FILE, index=False)
    print(f"Salvato: {LAYER_MAPPING_FILE}")


Checkpoint trovato: 4 layer già indotti.


In [17]:
# ── 5.3 Riepilogo dei layer indotti ──────────────────────────────────────────
print()
print("=" * 60)
print("LAYER TROVATI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    print(f"  [{row['layer_rank']}] {row['layer_name']}")
    print(f"      {str(row['layer_description'])[:200]}")
    print(f"      layer_id: {row['layer_id']}")
    print()
print(f"Totale layer: {len(layer_mapping_df)}")



LAYER TROVATI
  [1] Foundational scope and principles
      This layer contains the most general norms: scope clauses, core concepts, basic principles, and high-level policy orientations that define the reach and architecture of the regime. Provisions here do 
      layer_id: foundational_scope_and_principles

  [2] Structural and enabling rules
      This layer establishes the institutional and normative framework for implementation: delegations of power, organizational arrangements, internal responsibilities, and rules that channel how the regime
      layer_id: structural_and_enabling_rules

  [3] Operational application rules
      This layer translates the framework into concrete operative rules for defined procedures, categories of contracts, and specific legal effects. Provisions here regulate how a mechanism works in practic
      layer_id: operational_application_rules

  [4] Technical implementation details
      This layer contains the most detailed and execution-oriented p

## 7. Fase C — Distribuzione Percentuale per Articolo (LLM 3)

Con i layer noti, per ogni **articolo** di ogni atto l'LLM produce una distribuzione
percentuale del contenuto tra i layer emersi.

Il risultato per ogni atto è la matrice **articoli × layer** (valori = %) che alimenta
la heatmap nell'applicazione.

> **Perché solo gli articoli?** I considerando hanno funzione giustificativa e retorica:
> spesso coprono più livelli intenzionalmente per costruire l'argomentazione legale.
> La varianza dei considerando riflette struttura retorica, non patologia.
> Gli articoli hanno funzione prescrittiva — la loro ibridità è il segnale diagnostico.

In [18]:
# Ricarica layer mapping (può essere eseguita anche senza rieseguire B)
layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE).sort_values('layer_rank').reset_index(drop=True)

# layer_list: dict diretti dal CSV — espongono layer_name, layer_description,
# layer_id, layer_rank (chiavi usate dal prompt di Fase C).
# Aggiungo alias 'name'/'description'/'rank' per backward-compat con eventuali
# letture altrove nel notebook.
layer_list = []
for _, row in layer_mapping_df.iterrows():
    rec = row.to_dict()
    rec['name']        = rec['layer_name']
    rec['description'] = rec['layer_description']
    rec['rank']        = int(rec['layer_rank'])
    layer_list.append(rec)

layer_names = [l['layer_name'] for l in layer_list]

# Colonne CSV: prefisso pct__ seguito dal layer_id (slug)
pct_cols     = ['pct__' + l['layer_id'] for l in layer_list]
col_to_layer = {f'pct__{l["layer_id"]}': l['layer_name'] for l in layer_list}

# Solo gli articoli
articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()

print(f"Layer trovati: {len(layer_list)}")
for l in layer_list:
    print(f"  [{l['rank']}] {l['layer_name']}  (layer_id: {l['layer_id']})")
print()
print(f"Articoli da classificare: {len(articles_df):,}")
print(f"Atti coinvolti:           {articles_df['celex'].nunique():,}")


Layer trovati: 4
  [1] Foundational scope and principles  (layer_id: foundational_scope_and_principles)
  [2] Structural and enabling rules  (layer_id: structural_and_enabling_rules)
  [3] Operational application rules  (layer_id: operational_application_rules)
  [4] Technical implementation details  (layer_id: technical_implementation_details)

Articoli da classificare: 1,426
Atti coinvolti:           21


In [24]:
def build_prompt_percentage_distribution(testo, identificatore, layer_list, tema):
    layers_text = '\n'.join([
        f"  {l['layer_name']}: {l['layer_description']}"
        for l in layer_list
    ])
    return f"""You are an expert in legal analysis specializing in: {tema}

Read this article and:
1. Distribute its content as a percentage across the normative layers below.
2. For each layer with percentage > 0, quote 1-3 short excerpts from the text
   that justify the assignment and explain why in one sentence.

Layers:
{layers_text}

Rules:
- Percentages must sum to exactly 100.
- Assign 0 to layers not present.
- Quotes must be exact substrings of the article text.

Article {identificatore}:
{testo}

Reply ONLY with valid JSON (no backticks):
{{
  "Layer Name 1": X,
  "Layer Name 2": X,
  ...,
  "evidence": [
    {{"testo": "exact quote", "layer": "Layer Name", "motivo": "one sentence explanation"}},
    ...
  ]
}}"""


def parse_percentage_response(response_text, layer_names):
    """
    Parsa la risposta JSON e normalizza a somma 100.
    Restituisce None se il parsing fallisce.
    """
    try:
        clean  = response_text.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        return None

    values = {name: float(parsed.get(name, 0)) for name in layer_names}
    total  = sum(values.values())
    if total <= 0:
        return None
    if abs(total - 100) > 5:   # normalizza se la somma si discosta
        values = {k: v / total * 100 for k, v in values.items()}
    values['_evidence'] = parsed.get('evidence', [])
    return values


print("Funzioni Fase C definite.")


Funzioni Fase C definite.


In [30]:
%%time
import json as _json
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# ── Gestione checkpoint ────────────────────────────────────────────────────────
if os.path.exists(HEATMAP_CKPT_FILE):
    heatmap_done = pd.read_csv(HEATMAP_CKPT_FILE)

    def is_broken(row):
        if row.get('llm_status') != 'ok':
            return True
        try:
            ev = _json.loads(str(row.get('evidence', '[]')))
            return len(ev) == 0
        except Exception:
            return True

    def needs_evidence_only(row):
        """Ha percentuali ok ma evidence vuota — solo re-fetch evidence."""
        if row.get('llm_status') != 'ok':
            return False
        try:
            ev = _json.loads(str(row.get('evidence', '[]')))
            if len(ev) > 0:
                return False
            # Controlla che abbia almeno una pct > 0
            return any(row.get(c, 0) > 0 for c in pct_cols)
        except Exception:
            return False

    evidence_only_mask = heatmap_done.apply(needs_evidence_only, axis=1)
    broken_mask        = heatmap_done.apply(is_broken, axis=1) & ~evidence_only_mask
    good_mask          = ~broken_mask & ~evidence_only_mask

    print(f"Checkpoint: {len(heatmap_done):,} righe totali")
    print(f"  ok con evidence:      {int(good_mask.sum()):,}  → mantenute")
    print(f"  ok senza evidence:    {int(evidence_only_mask.sum()):,}  → solo evidence")
    print(f"  da rifare completo:   {int(broken_mask.sum()):,}")

    heatmap_good         = heatmap_done[good_mask].reset_index(drop=True)
    heatmap_evidence_only = heatmap_done[evidence_only_mask].reset_index(drop=True)
    done_seg_ids         = set(heatmap_good['segment_id'])
    evidence_only_ids    = set(heatmap_evidence_only['segment_id'])
else:
    heatmap_good          = pd.DataFrame()
    heatmap_evidence_only = pd.DataFrame()
    done_seg_ids          = set()
    evidence_only_ids     = set()
    print("Nessun checkpoint heatmap — si parte da zero.")

articles_full     = articles_df[~articles_df['segment_id'].isin(done_seg_ids | evidence_only_ids)].copy()
articles_ev_only  = articles_df[articles_df['segment_id'].isin(evidence_only_ids)].copy()

print(f"Classificazione completa: {len(articles_full):,}")
print(f"Solo evidence:            {len(articles_ev_only):,}")
print()

# ── Prompt leggero per soli evidence ──────────────────────────────────────────
def build_prompt_evidence_only(testo, identificatore, row_done, layer_list):
    """Prompt minimale: conosce già le % e chiede solo le citazioni."""
    assigned = ', '.join(
        f'"{l["layer_name"]}": {round(row_done.get("pct__" + l["layer_id"], 0), 0):.0f}%'
        for l in layer_list if row_done.get('pct__' + l['layer_id'], 0) > 0
    )
    return f"""You are an expert in legal analysis.

The following article has already been classified across normative layers:
{assigned}

Your only task: for each layer with percentage > 0, quote 1-3 short verbatim 
excerpts from the article text that justify that classification.

Article {identificatore}:
{testo}

Reply ONLY with valid JSON (no backticks):
{{
  "evidence": [
    {{"testo": "exact short quote", "layer": "Layer Name", "motivo": "one sentence"}},
    ...
  ]
}}"""

# ── Worker functions ───────────────────────────────────────────────────────────
checkpoint_lock = Lock()
results_buffer  = []
n_ok_c = n_error_c = 0
counter = [0]  # lista per mutabilità nel thread

def save_checkpoint(new_batch):
    all_good = heatmap_good if not heatmap_good.empty else pd.DataFrame()
    parts    = [p for p in [all_good, pd.DataFrame(new_batch)] if not p.empty]
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    combined.to_csv(HEATMAP_CKPT_FILE, index=False)

def process_full(art):
    prompt = build_prompt_percentage_distribution(
        testo=art['testo'], identificatore=art['identificatore'],
        layer_list=layer_list, tema=TEMA_DESCRIZIONE,
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_C)
    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, layer_names)
        if distribution is None:
            status = 'parse_error'
    row = {
        'segment_id':  art['segment_id'],
        'celex':       art['celex'],
        'node_id':     art['node_id'],
        'articolo_id': art['identificatore'],
        'llm_status':  status,
        'evidence':    _json.dumps(distribution.get('_evidence', []), ensure_ascii=False)
                       if distribution else '[]',
    }
    for col, name in zip(pct_cols, layer_names):
        row[col] = round(distribution[name], 2) if distribution else 0.0
    return row, distribution is not None

def process_evidence_only(art):
    done_row = heatmap_evidence_only[
        heatmap_evidence_only['segment_id'] == art['segment_id']
    ].iloc[0].to_dict()
    prompt = build_prompt_evidence_only(
        art['testo'], art['identificatore'], done_row, layer_list
    )
    response_text, status = call_llm(client, prompt, 800)
    evidence = []
    if status == 'ok':
        try:
            clean    = response_text.replace('```json','').replace('```','').strip()
            evidence = _json.loads(clean).get('evidence', [])
        except Exception:
            status = 'parse_error'
    # Aggiorna la riga esistente con l'evidence
    row = done_row.copy()
    row['evidence']   = _json.dumps(evidence, ensure_ascii=False)
    row['llm_status'] = 'ok' if evidence else 'parse_error'
    return row, bool(evidence)

# ── Esecuzione parallela ───────────────────────────────────────────────────────
MAX_WORKERS    = 8
CKPT_EVERY     = CHECKPOINT_EVERY
total_c        = len(articles_full) + len(articles_ev_only)
all_results    = []

def run_parallel(articles, worker_fn, label):
    global n_ok_c, n_error_c
    futures = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for _, art in articles.iterrows():
            futures[executor.submit(worker_fn, art.to_dict())] = art['segment_id']

        for i, future in enumerate(as_completed(futures)):
            row, success = future.result()
            all_results.append(row)
            if success:
                n_ok_c += 1
            else:
                n_error_c += 1

            done_total = len(all_results)
            if done_total % CKPT_EVERY == 0 or done_total == total_c:
                with checkpoint_lock:
                    save_checkpoint(all_results)
                print(f"  [{done_total:>5}/{total_c}]  {done_total/total_c*100:5.1f}%"
                      f"   ok: {n_ok_c}   errori: {n_error_c}  [{label}]")

print("── Classificazione completa ──────────────────────────────────────────────")
run_parallel(articles_full, process_full, "full")

print("── Solo evidence ─────────────────────────────────────────────────────────")
run_parallel(articles_ev_only, process_evidence_only, "evidence-only")

print()
print("=" * 50)
print(f"FASE C — ok: {n_ok_c:,}   errori: {n_error_c:,}")
print("=" * 50)

Checkpoint: 800 righe totali
  ok con evidence:      0  → mantenute
  ok senza evidence:    789  → solo evidence
  da rifare completo:   11
Classificazione completa: 637
Solo evidence:            789

── Classificazione completa ──────────────────────────────────────────────
  [  100/1426]    7.0%   ok: 94   errori: 6  [full]
  [  200/1426]   14.0%   ok: 193   errori: 7  [full]
  [  300/1426]   21.0%   ok: 292   errori: 8  [full]
  [  400/1426]   28.1%   ok: 390   errori: 10  [full]
  [  500/1426]   35.1%   ok: 490   errori: 10  [full]
  [  600/1426]   42.1%   ok: 589   errori: 11  [full]
── Solo evidence ─────────────────────────────────────────────────────────
  [  700/1426]   49.1%   ok: 685   errori: 15  [evidence-only]
  [  800/1426]   56.1%   ok: 776   errori: 24  [evidence-only]
  [  900/1426]   63.1%   ok: 871   errori: 29  [evidence-only]
  [ 1000/1426]   70.1%   ok: 970   errori: 30  [evidence-only]
  [ 1100/1426]   77.1%   ok: 1070   errori: 30  [evidence-only]
  [ 1200/1426

In [31]:
# Salva heatmap finale
heatmap_final = pd.read_csv(HEATMAP_CKPT_FILE)
heatmap_final.rename(columns={'celex': 'id'}).to_csv(NODES_HEATMAP_FILE, index=False)

print(f"Salvato: {NODES_HEATMAP_FILE}")
print(f"Righe: {len(heatmap_final):,}  |  Colonne pct: {pct_cols}")
print()

ok_mask = heatmap_final['llm_status'] == 'ok'
print("Distribuzione media % per layer (articoli ok):")
for col in pct_cols:
    mean_pct = heatmap_final.loc[ok_mask, col].mean()
    bar = '█' * int(mean_pct / 2)
    print(f"  {col_to_layer[col][:45]:.<46} {mean_pct:5.1f}%  {bar}")

Salvato: ..\data\output\appalti_it\nodes_heatmap.csv
Righe: 1,426  |  Colonne pct: ['pct__foundational_scope_and_principles', 'pct__structural_and_enabling_rules', 'pct__operational_application_rules', 'pct__technical_implementation_details']

Distribuzione media % per layer (articoli ok):
  Foundational scope and principles.............  25.5%  ████████████
  Structural and enabling rules.................  19.6%  █████████
  Operational application rules.................  29.7%  ██████████████
  Technical implementation details..............  25.2%  ████████████


## 7b. Fase C2 — Entropia Lamfalussy per Articolo

Calcola lo **score di ibridità Lamfalussy** partendo dall'output di Fase A2.
Nessuna chiamata LLM — è una trasformazione puramente computazionale.

Per ogni atto produce:
- `hybridity_lamf_score` — entropia media degli articoli su L1–L4 (metrica principale)
- `dominant_lamf` — livello Lamfalussy dominante

**Input**: `segments_lamfalussy.csv` (tutti i segmenti, filtrati agli articoli)

**Output**: `nodes_lamfalussy.csv` (articoli con distribuzione L1–L4, per le visualizzazioni)

In [32]:
# ── Fase C2: entropia Lamfalussy per articolo ────────────────────
if not os.path.exists(SEGMENTS_LAMF2_FILE):
    print(f"  {SEGMENTS_LAMF2_FILE} non trovato — esegui prima Fase A2 (sezione 4b).")
else:
    segments_lamf = pd.read_csv(SEGMENTS_LAMF2_FILE)

    # Filtra ai soli articoli (come Fase C sui layer emersi)
    articles_lamf = segments_lamf[
        (segments_lamf['llm_status'] == 'ok') &
        (segments_lamf['segment_id'].isin(set(articles_df['segment_id'])))
    ].copy()

    # Assicura che le colonne LAMF_COLS esistano
    for col in LAMF_COLS:
        if col not in articles_lamf.columns:
            articles_lamf[col] = 0.0

    # Rinomina come nodes_lamfalussy.csv (compatibilità con le celle di visualizzazione)
    # aggiunge articolo_id per allineamento con heatmap_ok
    articles_lamf['articolo_id'] = articles_lamf['identificatore']
    articles_lamf.to_csv(NODES_LAMFALUSSY2_FILE, index=False)

    print(f"Salvato: {NODES_LAMFALUSSY2_FILE}")
    print(f"Articoli: {len(articles_lamf):,}  |  Atti: {articles_lamf['celex'].nunique():,}")
    print()
    print("Distribuzione media % per livello Lamfalussy (articoli):")
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
        mean_pct = articles_lamf[col].mean()
        bar      = '█' * int(mean_pct / 2)
        print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")


Salvato: ..\data\output\appalti_it\nodes_lamfalussy2.csv
Articoli: 1,425  |  Atti: 21

Distribuzione media % per livello Lamfalussy (articoli):
  Level 1 — Framework principles................  42.1%  █████████████████████
  Level 2 — Operational rules...................  57.9%  ████████████████████████████


## 8. Fase D — Score di Ibridità per Atto

Lo score di ibridità misura quanto gli articoli di un atto variano nel loro livello
gerarchico. Un atto **puro** ha tutti gli articoli concentrati sullo stesso layer.
Un atto **ibrido** ha articoli che spaziano su layer molto diversi.

**Metrica: entropia di Shannon normalizzata per articolo**

$$H(a) = -\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)$$

Lo score dell'atto è la **media delle entropie dei propri articoli**.
Zero = tutti gli articoli sono monofunzionali. Uno = distribuzione uniforme su tutti i layer.

In [33]:
def entropy_norm(row, pct_cols):
    """Entropia di Shannon normalizzata (0=puro, 1=uniforme)."""
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in pct_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return 0.0
    probs = probs / s
    raw   = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(pct_cols)) if len(pct_cols) > 1 else 1.0
    return float(raw / maxH)


def dominant_layer(row, pct_cols, col_to_layer):
    best = max(pct_cols, key=lambda c: float(row.get(c, 0)))
    return col_to_layer.get(best, best)


# ── Articoli classificati correttamente (layer emersi) ────────────────────────
heatmap_ok = heatmap_final[heatmap_final['llm_status'] == 'ok'].copy()

# Entropia su layer emersi — metrica secondaria
heatmap_ok['entropy_layers'] = heatmap_ok.apply(
    lambda r: entropy_norm(r, pct_cols), axis=1
)
heatmap_ok['dominant_layer'] = heatmap_ok.apply(
    lambda r: dominant_layer(r, pct_cols, col_to_layer), axis=1
)

# ── Lamfalussy: definizione colonne (self-contained) ──────────────────────────
# Definite qui per garantire disponibilità anche se la cella 7b non è stata eseguita
_LAMF_KEYS_D = ['L1', 'L2']
_LAMF_COLS_D = [f'lamf_{k}' for k in _LAMF_KEYS_D]

# Merge entropia Lamfalussy in heatmap_ok ────────────────────────────────────
if os.path.exists(NODES_LAMFALUSSY2_FILE):
    lamfalussy_final = pd.read_csv(NODES_LAMFALUSSY2_FILE)
    lamf_ok = lamfalussy_final[lamfalussy_final['llm_status'] == 'ok'].copy()

    # Assicura che le colonne Lamfalussy esistano (gestisce run parziali)
    for col in _LAMF_COLS_D:
        if col not in lamf_ok.columns:
            lamf_ok[col] = 0.0

    lamf_ok['lamf_entropy'] = lamf_ok.apply(
        lambda r: entropy_norm(r, _LAMF_COLS_D), axis=1
    )
    lamf_ok['dominant_lamf'] = lamf_ok.apply(
        lambda r: max(_LAMF_COLS_D, key=lambda c: float(r.get(c, 0))).replace('lamf_', ''),
        axis=1
    )

    # Merge per-articolo in heatmap_ok: disponibile per le celle di visualizzazione
    heatmap_ok = heatmap_ok.merge(
        lamf_ok[['segment_id', 'lamf_entropy', 'dominant_lamf'] + _LAMF_COLS_D],
        on='segment_id', how='left'
    )
    heatmap_ok['lamf_entropy'] = heatmap_ok['lamf_entropy'].fillna(0.0)
    lamf_available = True
    print(f"Lamfalussy mergato in heatmap_ok: {lamf_ok['lamf_entropy'].notna().sum():,} articoli")
else:
    heatmap_ok['lamf_entropy']  = heatmap_ok['entropy_layers']   # fallback
    heatmap_ok['dominant_lamf'] = heatmap_ok['dominant_layer']
    lamf_available = False
    print(f"  {NODES_LAMFALUSSY2_FILE} non trovato — uso layer emersi come fallback per lamf_entropy")

# ── Aggregazione per atto ──────────────────────────────────────────────────────
def agg_atto(group):
    return pd.Series({
        # ── primario: Lamfalussy (o fallback su layer emersi) ─────────────────
        'hybridity_score':           group['lamf_entropy'].mean(),
        'hybridity_std':             group['lamf_entropy'].std(),
        'hybridity_max':             group['lamf_entropy'].max(),
        'most_hybrid_article':       (group.nlargest(1, 'lamf_entropy')['articolo_id'].iloc[0]
                                       if len(group) > 0 else ''),
        'dominant_lamf':             (group['dominant_lamf'].mode().iloc[0]
                                       if len(group) > 0 else ''),
        # ── secondario: layer emersi ──────────────────────────────────────────
        'hybridity_layers_score':    group['entropy_layers'].mean(),
        'hybridity_layers_std':      group['entropy_layers'].std(),
        'hybridity_layers_max':      group['entropy_layers'].max(),
        'dominant_layer':            (group['dominant_layer'].mode().iloc[0]
                                       if len(group) > 0 else ''),
        'dominant_layer_pct':        (group['dominant_layer'].value_counts().iloc[0] / len(group) * 100
                                       if len(group) > 0 else 0.0),
        'n_articles':                len(group),
    })

hybridity_df = heatmap_ok.groupby('celex').apply(agg_atto).reset_index()

# Aggiunge metadati dal nodo originale
meta_cols  = [c for c in ['Id', 'Label', 'title', 'LegalType', 'Year', 'PipelineLevel']
              if c in nodes.columns]
nodes_meta = nodes[meta_cols].copy()
nodes_meta = nodes_meta.rename(columns={'Label': 'celex'}) if 'Label' in nodes_meta.columns else nodes_meta

hybridity_df = hybridity_df.merge(nodes_meta, on='celex', how='left')
hybridity_df = hybridity_df.drop_duplicates(subset=['celex'], keep='first')
hybridity_df = hybridity_df.sort_values('hybridity_score', ascending=False)
hybridity_df.rename(columns={'celex': 'id'}).to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f"Salvato: {NODES_HYBRIDITY_FILE}")
print(f"Atti analizzati: {len(hybridity_df):,}")
print()
score_label = "Lamfalussy" if lamf_available else "Layer emersi (fallback)"
desc = hybridity_df['hybridity_score'].describe()
print(f"Statistiche hybridity_score ({score_label}):")
print(f"  Media:   {desc['mean']:.4f}")
print(f"  Mediana: {desc['50%']:.4f}")
print(f"  Max:     {desc['max']:.4f}")
print(f"  Std:     {desc['std']:.4f}")
print()
print("Top 5 atti più ibridi:")
for _, row in hybridity_df.head(5).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {celex_label:<20}  score={row['hybridity_score']:.3f}  "
          f"dominant={row['dominant_lamf']}  "
          f"(layers={row['hybridity_layers_score']:.3f})")


Lamfalussy mergato in heatmap_ok: 1,425 articoli
Salvato: ..\data\output\appalti_it\nodes_hybridity.csv
Atti analizzati: 21

Statistiche hybridity_score (Lamfalussy):
  Media:   0.7206
  Mediana: 0.7268
  Max:     0.7993
  Std:     0.0575

Top 5 atti più ibridi:
  l_55_2019             score=0.799  dominant=L2  (layers=0.834)
  l_120_2020            score=0.792  dominant=L2  (layers=0.832)
  l_114_2014            score=0.790  dominant=L2  (layers=0.880)
  l_108_2021            score=0.779  dominant=L2  (layers=0.796)
  dlgs_36_2023          score=0.777  dominant=L2  (layers=0.793)


## 9. Diagnostica

In [34]:
# Riepilogo layer indotti + (se disponibile) conteggio articoli con layer dominante.
print("=" * 60)
print("LAYER EMERSI")
print("=" * 60)

# Conteggio articoli per layer dominante (calcolato dalla heatmap se presente)
dom_counts = {}
if os.path.exists(NODES_HEATMAP_FILE):
    _hm = pd.read_csv(NODES_HEATMAP_FILE)
    _hm_ok = _hm[_hm['llm_status'] == 'ok'] if 'llm_status' in _hm.columns else _hm
    if len(_hm_ok) and len(pct_cols):
        _hm_ok = _hm_ok.copy()
        _hm_ok['_dom'] = _hm_ok[pct_cols].idxmax(axis=1)
        for col, n in _hm_ok['_dom'].value_counts().items():
            layer_name = col_to_layer.get(col, col)
            dom_counts[layer_name] = int(n)
    total_arts = len(_hm_ok)
else:
    total_arts = 0

for _, row in layer_mapping_df.iterrows():
    name = row['layer_name']
    n    = dom_counts.get(name, 0)
    pct  = (n / total_arts * 100) if total_arts > 0 else 0.0
    bar  = '█' * int(pct)
    print(f"[{row['layer_rank']:>2}] {name}")
    print(f"     layer_id: {row['layer_id']}")
    if total_arts > 0:
        print(f"     Articoli con layer dominante: {n:,}  ({pct:.1f}%)  {bar}")
    print(f"     {str(row['layer_description'])[:200]}")
    print()


LAYER EMERSI
[ 1] Foundational scope and principles
     layer_id: foundational_scope_and_principles
     Articoli con layer dominante: 287  (20.6%)  ████████████████████
     This layer contains the most general norms: scope clauses, core concepts, basic principles, and high-level policy orientations that define the reach and architecture of the regime. Provisions here do 

[ 2] Structural and enabling rules
     layer_id: structural_and_enabling_rules
     Articoli con layer dominante: 216  (15.5%)  ███████████████
     This layer establishes the institutional and normative framework for implementation: delegations of power, organizational arrangements, internal responsibilities, and rules that channel how the regime

[ 3] Operational application rules
     layer_id: operational_application_rules
     Articoli con layer dominante: 549  (39.5%)  ███████████████████████████████████████
     This layer translates the framework into concrete operative rules for defined procedures, catego

## 10. Riepilogo Output

Verifica che tutti i file di output siano stati prodotti correttamente.

In [35]:
output_files = {
    'segments_descriptions.csv': SEGMENTS_DESC_FILE,
    'layer_mapping.csv':         LAYER_MAPPING_FILE,
    'nodes_heatmap.csv':         NODES_HEATMAP_FILE,
    'nodes_hybridity.csv':       NODES_HYBRIDITY_FILE,
    'segments_lamfalussy.csv':    SEGMENTS_LAMF2_FILE,
    'nodes_lamfalussy.csv':       NODES_LAMFALUSSY2_FILE,
}

print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

all_ok = True
for name, path in output_files.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        df_tmp  = pd.read_csv(path)
        print(f"  ✓ {name}")
        print(f"    Righe: {len(df_tmp):,}  |  Dim: {size_kb:.1f} KB")
        print(f"    Colonne: {list(df_tmp.columns)[:6]}{'...' if len(df_tmp.columns) > 6 else ''}")
    else:
        print(f"  ✗ {name} — FILE MANCANTE")
        all_ok = False
    print()

if all_ok:
    print("✓ Pipeline 04 completata.")
else:
    print("  Alcuni file mancano — rieseguire le celle corrispondenti.")

OUTPUT FILES
  ✓ segments_descriptions.csv
    Righe: 1,426  |  Dim: 4706.4 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'testo_originale']...

  ✓ layer_mapping.csv
    Righe: 4  |  Dim: 1.8 KB
    Colonne: ['layer_rank', 'layer_name', 'layer_description', 'layer_id']

  ✓ nodes_heatmap.csv
    Righe: 1,426  |  Dim: 2887.9 KB
    Colonne: ['segment_id', 'id', 'node_id', 'articolo_id', 'llm_status', 'evidence']...

  ✓ nodes_hybridity.csv
    Righe: 21  |  Dim: 9.2 KB
    Colonne: ['id', 'hybridity_score', 'hybridity_std', 'hybridity_max', 'most_hybrid_article', 'dominant_lamf']...

  ✓ segments_lamfalussy.csv
    Righe: 1,434  |  Dim: 1841.6 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'llm_status']...

  ✓ nodes_lamfalussy.csv
    Righe: 1,425  |  Dim: 1834.8 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'llm_status']...

✓ Pipeline 04 completata.


## Fase E — Export per il frontend HTML

Genera `HEATMAPS` (dizionario `id → articoli`) e patcha l'HTML self-contained
con tutti i dati della pipeline: layer labels, distribuzioni percentuali, campo `heatmap` nei nodi.

**Cambia `HTML_FILE`** per puntare al file corretto prima di eseguire.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE E — Export HEATMAPS + distribuzione Lamfalussy per il frontend HTML
# ══════════════════════════════════════════════════════════════════════════════

HTML_FILE = os.path.join('..', 'appalti_network_2_layers.html')

import re, math, json as _json

# ── 1. Layer labels da layer_mapping.csv ──────────────────────────────────────
layer_df = pd.read_csv(LAYER_MAPPING_FILE).sort_values('layer_rank')
layer_names_raw = layer_df['layer_name'].tolist()

def split_label(name):
    s = name.replace('_', ' ')
    words = s.split()
    if len(words) == 1:
        return s, ''
    half = len(s) // 2
    pos, best_split = 0, len(words) // 2
    for i, w in enumerate(words[:-1]):
        pos += len(w) + 1
        if pos >= half:
            best_split = i + 1
            break
    return ' '.join(words[:best_split]), ' '.join(words[best_split:])

l1_arr, l2_arr = [], []
for name in layer_names_raw:
    a, b = split_label(name)
    l1_arr.append(a)
    l2_arr.append(b)

print(f"Layer ({len(layer_names_raw)}):")
for i, name in enumerate(layer_names_raw):
    print(f"  {i:2d}  '{l1_arr[i]}' / '{l2_arr[i]}'")

# ── Aggiungi lamf per articolo a hm_df ───────────────────────────────────────
hm_df = pd.read_csv(NODES_HEATMAP_FILE)   # ← qui, prima del merge

# ── Aggiungi lamf per articolo a hm_df ───────────────────────────────────────
if os.path.exists(SEGMENTS_LAMF2_CKPT_FILE):
    lamf_art = pd.read_csv(SEGMENTS_LAMF2_CKPT_FILE)
    lamf_art = lamf_art[lamf_art['llm_status'] == 'ok'].copy()
    lamf_art = lamf_art.rename(columns={'identificatore': 'articolo_id'})
    if 'node_id' not in lamf_art.columns:
        lamf_art = lamf_art.rename(columns={'celex': 'node_id'})
    hm_df = hm_df.drop(columns=[c for c in hm_df.columns if c.startswith('lamf_L')], errors='ignore')
    lamf_cols_to_add = [c for c in lamf_art.columns if c.startswith('lamf_L')]
    hm_df = hm_df.merge(
        lamf_art[['node_id', 'articolo_id'] + lamf_cols_to_add],
        on=['node_id', 'articolo_id'], how='left'
    )
    print(f"lamf_L1 non-null: {hm_df['lamf_L1'].notna().sum()} / {len(hm_df)}")

# ── 2. HEATMAPS dict da nodes_heatmap.csv ────────────────────────────────────
id_col = 'node_id' if 'node_id' in hm_df.columns else ('id' if 'id' in hm_df.columns else 'celex')
pct_cols_hm = [c for c in hm_df.columns if c.startswith('pct__')]
n_layers = len(pct_cols_hm)

HEATMAPS = {}
for act_id, grp in hm_df.groupby(id_col):
    ok_grp = grp[grp['llm_status'] == 'ok'].copy()
    if ok_grp.empty:
        continue
    articles = []
    for _, row in ok_grp.iterrows():
        vals = [float(row[c]) for c in pct_cols_hm]
        H = 0.0
        for v in vals:
            p = v / 100.0
            if p > 1e-9:
                H -= p * math.log2(p)
        H_norm = round(H / math.log2(n_layers), 3) if n_layers > 1 else 0.0
        
        art_entry = {'id': str(row['articolo_id']), 'H': H_norm,
             'vals': [round(v, 1) for v in vals]}
        # aggiungi lamf se disponibile nel merge
        for lk in ['L1','L2']:
            col = f'lamf_{lk}'
            if col in row and pd.notna(row[col]):
                art_entry.setdefault('lamf', {})[lk] = round(float(row[col]), 1)
        # Shannon H normalizzata sui livelli Lamfalussy
        if 'lamf' in art_entry:
            lv = list(art_entry['lamf'].values())
            s_lv = sum(lv)
            if s_lv > 0:
                probs_lv = [v / s_lv for v in lv]
                raw_h_lv = -sum(p * math.log2(p) for p in probs_lv if p > 1e-9)
                art_entry['H_lamf'] = round(raw_h_lv / math.log2(max(len(lv), 2)), 3)
        articles.append(art_entry)

    if articles:
        HEATMAPS[str(act_id)] = articles

print(f"\nHEATMAPS: {len(HEATMAPS)} atti  |  "
      f"{sum(len(v) for v in HEATMAPS.values())} articoli totali")

# ── 3. Distribuzione Lamfalussy per atto (media articoli) ─────────────────────
LAMF_DIST = {}
if os.path.exists(NODES_LAMFALUSSY2_FILE):
    lamf_df  = pd.read_csv(NODES_LAMFALUSSY2_FILE)
    lamf_cols = [c for c in lamf_df.columns if c.startswith('lamf_L')]
    lamf_ok  = lamf_df[lamf_df['llm_status'] == 'ok']
    celex_col = 'celex' if 'celex' in lamf_ok.columns else 'id'
    if lamf_cols and not lamf_ok.empty:
        grp = lamf_ok.groupby(celex_col)[lamf_cols].mean().round(1)
        for act_id, row in grp.iterrows():
            LAMF_DIST[str(act_id)] = {k.replace('lamf_', ''): float(v)
                                       for k, v in row.items()}
    print(f"LAMF_DIST: {len(LAMF_DIST)} atti")
else:
    print("WARN: nodes_lamfalussy.csv non trovato — lamf non aggiunto ai nodi")

# ── 3b. Evidence per articolo da checkpoint A2 ────────────────────────────────
EVIDENCE = {}
if os.path.exists(SEGMENTS_LAMF2_CKPT_FILE):
    ckpt = pd.read_csv(SEGMENTS_LAMF2_CKPT_FILE)
    ckpt_ok = ckpt[ckpt['llm_status'] == 'ok']
    ck_id   = 'celex' if 'celex' in ckpt_ok.columns else 'id'
    for (celex, idf), grp in ckpt_ok.groupby([ck_id, 'identificatore']):
        ev_list = []
        for _, row in grp.iterrows():
            raw = row.get('evidence', '[]')
            try:
                ev_list.extend(_json.loads(raw) if isinstance(raw, str) else [])
            except Exception:
                pass
        if ev_list:
            EVIDENCE[(str(celex), str(idf))] = ev_list
    print(f"EVIDENCE: {len(EVIDENCE)} articoli con evidence")
else:
    print("WARN: checkpoint A2 non trovato — evidence non disponibile")

# ── 3c. Testi articoli da nodes_texts.csv ────────────────────────────────────
ART_TEXTS = {}
if os.path.exists(INPUT_FILE):
    texts_df = pd.read_csv(INPUT_FILE)
    id_col_t = 'Id' if 'Id' in texts_df.columns else 'celex'
    for _, row in texts_df.iterrows():
        celex = str(row[id_col_t])
        raw   = row.get('segments', '')
        if not raw or str(raw) in ('nan', '[]', ''):
            continue
        try:
            for seg in _json.loads(str(raw)):
                if seg.get('tipo') == 'articolo':
                    ART_TEXTS[(celex, str(seg.get('identificatore', '')))] = seg.get('testo', '')
        except Exception:
            pass
    print(f"ART_TEXTS: {len(ART_TEXTS)} articoli con testo")

# ── 3d. Arricchisce HEATMAPS con testo ed evidence ───────────────────────────
for celex, articles in HEATMAPS.items():
    for art in articles:
        key = (celex, art['id'])
        art['ev']  = EVIDENCE.get(key, [])
        art['txt'] = ART_TEXTS.get(key, '')

ev_count = sum(1 for arts in HEATMAPS.values() for a in arts if a.get('ev'))
print(f"Articoli con ev in HEATMAPS: {ev_count}")

for celex, articles in HEATMAPS.items():
    for art in articles:
        total = sum(art['vals'])
        if total > 0 and abs(total - 100) > 0.1:
            art['vals'] = [round(v / total * 100, 1) for v in art['vals']]

import re as _re
def art_sort_key(a):
    m = _re.match(r'(\d+)', str(a['id']))
    return int(m.group(1)) if m else 9999
HEATMAPS = {celex: sorted(arts, key=art_sort_key) for celex, arts in HEATMAPS.items()}

# ── 4. H Lamfalussy per atto da nodes_hybridity.csv ──────────────────────────
HYB_MAP = {}
if os.path.exists(NODES_HYBRIDITY_FILE):
    hyb_df   = pd.read_csv(NODES_HYBRIDITY_FILE)
    hid_col  = 'celex' if 'celex' in hyb_df.columns else 'id'
    for _, row in hyb_df.iterrows():
        HYB_MAP[str(row[hid_col])] = round(float(row.get('hybridity_score', 0)), 3)
    print(f"HYB_MAP:   {len(HYB_MAP)} atti")

# ── 5. Save heatmaps.json (proper JSON serialization — no syntax errors) ─────
import os as _os

JSON_FILE = _os.path.join(_os.path.dirname(HTML_FILE), 'heatmaps.json')

with open(JSON_FILE, 'w', encoding='utf-8') as f:
    _json.dump(HEATMAPS, f, ensure_ascii=False)

print(f"\n✓ Salvato {JSON_FILE}  ({_os.path.getsize(JSON_FILE)//1024} KB)")

# ── 6. Patch HTML ─────────────────────────────────────────────────────────────
with open(HTML_FILE, 'r', encoding='utf-8') as f:
    html = f.read()

# 6a. Replace inline HEATMAPS const with a fetch()-based async loader.
#     The loader wraps ALL visualization logic that depends on HEATMAPS inside
#     the .then() callback so it runs only after the JSON is available.
FETCH_SNIPPET = (
    "fetch('heatmaps.json')\n"
    "  .then(r => r.json())\n"
    "  .then(HEATMAPS => {\n"
    "    // HEATMAPS loaded from heatmaps.json\n"
    "    window.__HEATMAPS__ = HEATMAPS;\n"
    "  })\n"
    "  .catch(e => console.error('Failed to load heatmaps.json:', e));"
)

# Remove any existing inline HEATMAPS const (may be large, use non-greedy pattern)
html, n_hm = re.subn(
    r'const HEATMAPS\s*=\s*\{[\s\S]*?\};',
    FETCH_SNIPPET,
    html,
    count=1
)
if n_hm == 0:
    # Also try the old HMxxx pattern from prior notebook versions
    html, n_hm = re.subn(r'const HM\w+\s*=\s*\[[\s\S]*?\];', FETCH_SNIPPET, html, count=1)
print(f"HEATMAPS inline → fetch(): {n_hm} sostituzioni")

# Patch all HEATMAPS usages in visualization JS to use window.__HEATMAPS__
html = html.replace('HEATMAPS[', 'window.__HEATMAPS__[')
html = html.replace('HEATMAPS)', 'window.__HEATMAPS__)')
html = html.replace('HEATMAPS,', 'window.__HEATMAPS__,')
html = html.replace('HEATMAPS.', 'window.__HEATMAPS__.')
html = html.replace(' HEATMAPS ', ' window.__HEATMAPS__ ')

# 6b. L1 / L2
html, n = re.subn(r'const L1\s*=\s*\[.*?\];',
                   'const L1     = ' + _json.dumps(l1_arr) + ';', html)
print(f"L1  sostituito: {n}")
html, n = re.subn(r'const L2\s*=\s*\[.*?\];',
                   'const L2     = ' + _json.dumps(l2_arr) + ';', html)
print(f"L2  sostituito: {n}")

# 6c. NODES: heatmap, lamf, H
def patch_nodes(html, heatmap_ids, lamf_dist, hyb_map):
    m = re.search(r'(const NODES\s*=\s*)(\[[\s\S]*?\]);', html)
    if not m:
        print("WARN: const NODES non trovato")
        return html
    nodes = _json.loads(m.group(2))
    for nd in nodes:
        nid = nd['id']
        nd['heatmap'] = 'computed' if nid in heatmap_ids else nd.pop('heatmap', None) or None
        if nd.get('heatmap') is None:
            nd.pop('heatmap', None)
        if nid in lamf_dist:
            nd['lamf'] = lamf_dist[nid]
        else:
            nd.pop('lamf', None)
        if nid in hyb_map:
            nd['H'] = hyb_map[nid]
    new_block = m.group(1) + _json.dumps(nodes, ensure_ascii=False) + ';'
    lamf_n = sum(1 for nd in nodes if 'lamf' in nd)
    hm_n   = sum(1 for nd in nodes if nd.get('heatmap') == 'computed')
    print(f"NODES: {hm_n} con heatmap  |  {lamf_n} con lamf")
    return html[:m.start()] + new_block + html[m.end():]

html = patch_nodes(html, set(HEATMAPS.keys()), LAMF_DIST, HYB_MAP)

with open(HTML_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"\n✓ {HTML_FILE}")

Layer (4):
   0  'Foundational scope' / 'and principles'
   1  'Structural and' / 'enabling rules'
   2  'Operational application' / 'rules'
   3  'Technical implementation' / 'details'
lamf_L1 non-null: 1425 / 1426

HEATMAPS: 21 atti  |  1390 articoli totali
LAMF_DIST: 21 atti
EVIDENCE: 1433 articoli con evidence
ART_TEXTS: 1426 articoli con testo
Articoli con ev in HEATMAPS: 1389
HYB_MAP:   21 atti

✓ Salvato ..\heatmaps.json  (5619 KB)
HEATMAPS inline → fetch(): 0 sostituzioni
L1  sostituito: 1
L2  sostituito: 1
NODES: 20 con heatmap  |  20 con lamf

✓ ..\appalti_network_2_layers.html
